# 🚗 Waze User Churn Analysis Report

This notebook covers step-by-step data cleaning, Exploratory Data Analysis (EDA), and behavioral insights. The goal is to prepare the Waze dataset for Machine Learning modeling 🧠.

## 📂 1. Environment Setup & Ingestion
### Step 1: Library Ingestion and Setup
We start by importing the core libraries for data processing and interactive visualization.

In [1]:
# Data Processing
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

# Warnings
import warnings
warnings.filterwarnings('ignore')

### Step 2: Data Ingestion and Structural Auditing
Loading the dataset from the CSV file and checking its size and columns.

In [2]:
df = pd.read_csv('../data/row/waze_dataset.csv')
df.head()

,ID,label,sessions,drives,total_sessions,n_days_after_onboarding,total_navigations_fav1,total_navigations_fav2,driven_km_drives,duration_minutes_drives,activity_days,driving_days,device
0,0,retained,283,226,296.748273,2276,208,0,2628.845068,1985.775061,28,19,Android
1,1,retained,133,107,326.896596,1225,19,64,13715.920550,3160.472914,13,11,iPhone
2,2,retained,114,95,135.522926,2651,0,0,3059.148818,1610.735904,14,8,Android
3,3,retained,49,40,67.589221,15,322,7,913.591123,587.196542,7,3,iPhone
4,4,retained,84,68,168.247020,1562,166,5,3950.202008,1219.555924,27,18,Android


In [3]:
df.shape

(14999, 13)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14999 entries, 0 to 14998
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       14999 non-null  int64  
 1   label                    14299 non-null  object 
 2   sessions                 14999 non-null  int64  
 3   drives                   14999 non-null  int64  
 4   total_sessions           14999 non-null  float64
 5   n_days_after_onboarding  14999 non-null  int64  
 6   total_navigations_fav1   14999 non-null  int64  
 7   total_navigations_fav2   14999 non-null  int64  
 8   driven_km_drives         14999 non-null  float64
 9   duration_minutes_drives  14999 non-null  float64
 10  activity_days            14999 non-null  int64  
 11  driving_days             14999 non-null  int64  
 12  device                   14999 non-null  object 
dtypes: float64(3), int64(8), object(2)
memory usage: 1.5+ MB


> **❓ Exploratory Question:** Why is `total_sessions` saved as a float (decimal number) instead of an integer (whole number)?
> 
> **🛠️ Action Plan:** We will round it and change it to an integer type in the next section.

### Step 3: Missing Value Profiling
Checking for empty or null cells in our columns.

In [5]:
df.isnull().sum()

ID                           0
label                      700
sessions                     0
drives                       0
total_sessions               0
n_days_after_onboarding      0
total_navigations_fav1       0
total_navigations_fav2       0
driven_km_drives             0
duration_minutes_drives      0
activity_days                0
driving_days                 0
device                       0
dtype: int64

> **⚠️ Data Quality Note:** The target column `label` has exactly **700 missing values**. Since this is the main variable we want to predict, we must remove these rows.

## 🧹 2. Data Quality Cleaning & Refinement
### Step 4: Cleaning and Formatting
Fixing data types, rounding metrics, and dropping rows with missing target labels.

In [6]:
# converting total_sessions to integer
df['total_sessions'] = df['total_sessions'].round().astype(int)

In [7]:
# rounding driven_km and duration_minutes to 1 decimal place
df['driven_km_drives'] = df['driven_km_drives'].round(1)
df['duration_minutes_drives'] = df['duration_minutes_drives'].round(1)

In [8]:
# remove missing values in label column
df = df.dropna(subset=['label'])

In [9]:
df

,ID,label,sessions,drives,total_sessions,n_days_after_onboarding,total_navigations_fav1,total_navigations_fav2,driven_km_drives,duration_minutes_drives,activity_days,driving_days,device
0,0,retained,283,226,297,2276,208,0,2628.8,1985.8,28,19,Android
1,1,retained,133,107,327,1225,19,64,13715.9,3160.5,13,11,iPhone
2,2,retained,114,95,136,2651,0,0,3059.1,1610.7,14,8,Android
3,3,retained,49,40,68,15,322,7,913.6,587.2,7,3,iPhone
4,4,retained,84,68,168,1562,166,5,3950.2,1219.6,27,18,Android
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14994,14994,retained,60,55,208,140,317,0,2890.5,2186.2,25,17,iPhone
14995,14995,retained,42,35,188,2505,15,10,4062.6,1208.6,25,20,Android
14996,14996,retained,273,219,422,1873,17,0,3097.8,1031.3,18,17,iPhone
14997,14997,churned,149,120,181,3150,45,0,4051.8,254.2,6,6,iPhone


In [10]:
df.shape

(14299, 13)

## 📊 3. Exploratory Data Analysis (EDA)
### Step 5: Univariate Outlier and Distribution Exploration

> **❓ Chart Question:** What is the percentage of users who left the app (Churned) versus users who stayed (Retained)?

In [11]:
df["label"].value_counts()

label
retained    11763
churned      2536
Name: count, dtype: int64

In [12]:
px.pie(df, names='label', title='Distribution of Churned vs Non-Churned Users')

> **💡 Insight A (User Split):** 
> The data shows that **17.7%** of the users left the app (2,536 users), while **82.3%** stayed (11,763 users). There is a small class imbalance that we need to handle during modeling.

> **❓ Chart Question:** How is the monthly `sessions` metric distributed? Are there any extreme values or outliers?

In [13]:
df['sessions'].min(),df['sessions'].max()

(0, 743)

In [14]:
px.box(df, x='sessions', title='Distribution & outliers of sessions')

> **💡 Insight B (Extreme Outliers):** 
> The box plot shows a long right tail with many points outside the top line. This means Waze has a group of **Power Users** who use the app very heavily (like professional drivers), making the data right-skewed.

> **❓ Chart Question:** Do other activity metrics show the same outlier behavior? How is the user base split by device type?

In [15]:
px.box(df, x='drives', title='Distribution & outliers of drives')

> **💡 Insight 3 (Drives Outliers):** 
> Just like sessions, the monthly `drives` data contains massive outliers on the right side[cite: 1], pointing to heavy app users.

> **❓ Chart Question 4:** What does the data spread look like for historical `total_sessions`?

In [16]:
px.box(df, x='total_sessions', title='Distribution & outliers of total_sessions')

> **💡 Insight 4 (Total Sessions Outliers):** 
> Historical total sessions contain heavy right-side outliers[cite: 1], meaning some drivers have a massive usage history on Waze.

> **❓ Chart Question 5:** Are there any outlier behaviors for the duration after onboarding (`n_days_after_onboarding`)?

In [17]:
px.box(df, x='n_days_after_onboarding', title='Distribution & outliers of n_days_after_onboarding')

> **💡 Insight 5 (Onboarding Distribution):** 
> Unlike the activity columns, onboarding days are evenly distributed with no extreme outliers.This represents a steady user sign-up history.

> **❓ Chart Question 6:** What is the spread of the total kilometers driven by our users (`driven_km_drives`)?

In [18]:
px.box(df, x='driven_km_drives', title='Distribution & outliers of driven_km_drives')


> **💡 Insight 6 (Driven KM Outliers):** 
> The kilometers driven metric displays huge outliers on the right side[cite: 1]. A subset of users drives extremely long distances.

> **❓ Chart Question 7:** How are the driving durations in minutes distributed among users (`duration_minutes_drives`)?

In [19]:
px.box(df, x='duration_minutes_drives', title='Distribution & outliers of duration_minutes_drives')


> **💡 Insight 7 (Duration Outliers):** 
> Driving duration heavily mirrors the distance metric, revealing huge outliers from users spending an immense amount of time on the road

> **❓ Chart Question 8:** How is the Waze user base split by device type?

In [20]:
px.pie(df, names='device', title='Distribution & outliers of device')


> **💡 Insight 8 (Device Breakdown):** 
> The pie chart shows that iPhone users form the clear majority of our driving user base compared to Android users.

## 🧠 4. Behavioral Churn Segmentation (Bivariate EDA)
We link active metrics with churn labels to analyze core segment behaviors.

> **❓ Chart Question 9:** Do users who churn log fewer average monthly sessions?

In [21]:
df.groupby('label')['sessions'].mean().reset_index()

,label,sessions
0,churned,87.238959
1,retained,79.197654


In [22]:
px.bar(df.groupby('label')['sessions'].mean().reset_index(), x='label', y='sessions', title='Average Sessions by Churn Label')

> **💡 Insight 9 (Sessions Comparison):** 
> Surprisingly, churned users show a *higher* average monthly session count (87.24) compared to retained users (79.19).

> **❓ Chart Question 10:** What is the average number of active days per month (`activity_days`) for both groups?

In [23]:
df.groupby('label')['activity_days'].mean().reset_index()

,label,activity_days
0,churned,9.644716
1,retained,16.816628


In [24]:
px.bar(df.groupby('label')['activity_days'].mean().reset_index(), x='label', y='activity_days', title='Average Activity Days by Churn Label')

> **💡 Insight 10 (The Activity Paradox):** 
> Even though churned users have more sessions, they open the app on far fewer days (~9.64 days) than retained users (~16.81 days)[cite: 1]. This major drop in consistency is our strongest churn indicator.

> **❓ Chart Question 11:** How does historical total session usage compare between the two groups?

In [25]:
df.groupby('label')['total_sessions'].mean().reset_index()

,label,total_sessions
0,churned,196.898659
1,retained,187.965655


In [26]:
px.bar(df.groupby('label')['total_sessions'].mean().reset_index(), x='label', y='total_sessions', title='Average Total Sessions by Churn Label')

> **💡 Insight 11 (Total Historical Sessions):** 
> Churned users show higher historical total sessions (196.90) than retained users (187.96), proving that intensive users still churn if daily habits drop.

> **❓ Chart Question 12:** Is user churn concentrated heavily on iPhone or Android devices?

In [27]:
df.groupby(['label', 'device'])['device'].count().reset_index(name='count_device')

,label,device,count_device
0,churned,Android,891
1,churned,iPhone,1645
2,retained,Android,4183
3,retained,iPhone,7580


In [28]:
px.bar(df.groupby(['label', 'device'])['device'].count().reset_index(name='count_device'), 
       x='label', 
       y='count_device', 
       color='device',
       text='count_device',
       title='Distribution of Users by Churn Label and Device')

> **💡 Insight 12 (Device Distribution):** 
> Churn rates are nearly identical: **17.83%** for iPhone and **17.56%** for Android. This points to a general user behavior pattern rather than technical device bugs.

## 🛣️ 5. Trip Efficiency Analysis
We evaluate patterns correlating driven kilometers against driving durations.

> **❓ Chart Question 13:** Is the correlation between driven distance and driving duration linear and logical across phone types?

In [29]:
px.scatter(df, 
       x='duration_minutes_drives', 
       y='driven_km_drives',
       color='device',
       title='Trip Efficiency: Driven KM vs. Duration Minutes',
       trendline='ols',
       trendline_color_override='green',
       )

> **💡 Insight 13 (Trip Efficiency):** 
> The scatter plot displays a tight linear trend line[cite: 1]. Driving metrics across both platforms show identical performance and logical consistency[cite: 1].

In [30]:
# save the cleaned dataset to build Ml model
df.to_csv('../data/cleaned/waze_dataset_cleaned.csv', index=False)

# 📊 Executive Summary: Waze User Churn Analysis

## 1️⃣ Chronological Summary of Analysis Steps

1. **Environment Setup & Library Ingestion** ⚙️
   * Imported essential data science libraries for data frame manipulation (`pandas`, `numpy`) and interactive visualizations (`plotly.express`, `matplotlib`, `seaborn`). 
   * Configured warning filters (`warnings.filterwarnings('ignore')`) to ensure clean report outputs.

2. **Data Ingestion & Structural Auditing** 📥
   * Loaded the primary source data (`waze_dataset.csv`), exposing an initial database shape of **14,999 rows and 13 columns**. 
   * Inspected column types using `.info()`, revealing that the discrete metric `total_sessions` was saved as a decimal float (`float64`) instead of an integer.

3. **Missing Value Profiling** 🔍
   * Executed a structural missing value scan (`df.isnull().sum()`), locating exactly **700 empty entries**. 
   * Classified that these missing cells were completely isolated inside the `label` feature (our target variable representing whether a user stayed or left).

4. **Data Quality Cleaning & Formatting** 🧹
   * Fixed column structural type mismatch by rounding and casting `total_sessions` to a whole integer (`int`).
   * Standardized spatial/time precision by rounding `driven_km_drives` and `duration_minutes_drives` to $1$ decimal place.
   * Dropped the 700 missing target label rows to guarantee clean inputs, yielding a final processed baseline of **14,299 clean rows**.

5. **Univariate Outlier Exploration** 📈
   * Programmed individual Plotly box plots for continuous activity columns (`sessions`, `drives`, `total_sessions`, `driven_km_drives`, `duration_minutes_drives`).
   * Observed strong right-side data skewness across all core activity parameters, exposing massive activity ranges.

6. **Bivariate Behavioral Churn Segmentation** 👥
   * Structured group-by aggregations and bar charts crossing the target retention status (`retained` vs. `churned`) with average app behaviors.
   * Crossed user device profiles (`Android` vs. `iPhone`) against churn volumes to discover system-level distribution trends.

7. **Trip Efficiency Analysis** 🛣️
   * Created a comprehensive trip dynamic scatter plot mapping driving distance ($Y$-axis) against driving time ($X$-axis).
   * Fitted an Ordinary Least Squares (OLS) linear trendline to confirm behavioral correlation across device platforms.

8. **Clean Data Asset Exportation** 💾
   * Exported the completed pipeline output to `../data/waze_dataset_cleaned.csv` to serve as a reliable, clean training set for predictive machine learning classification models.

---

## 2️⃣ Key Data-Driven Insights

### 💡 Insight A: Base Target Distribution
The dataset features a steady operational representation across both target retention groups:
* **Retained Users:** 11,763 drivers (**82.3%** of the user base)
* **Churned Users:** 2,536 drivers (**17.7%** of the user base)

### 💡 Insight B: The Activity Paradox (High Intensity vs. Low Consistency)
Cross-referencing behavioral summaries between classes revealed a counter-intuitive usage paradox:
* **Monthly Active Sessions:** Churned users log a *higher* monthly average (**87.24 sessions**) than retained users (**79.19 sessions**).
* **Historical Total Sessions:** Churned users show a *higher* lifetime count (**196.90 total sessions**) than retained users (**187.96 total sessions**).
* **Monthly Active Days:** Crucially, churned users are active only **9.64 days per month**, whereas retained users use the app on **16.81 days per month**.

> **Strategic Interpretation:** Churn is *not* caused by low usage intensity. High-risk users interact heavily and log more sessions when they use the app. However, they lack longitudinal consistency. A drop-off in monthly active days (`activity_days`) is the primary behavioral red flag indicating imminent churn risk.

### 💡 Insight C: Right-Skewed Skewness driven by Power Users
The box plots for activity, mileage, and driving minutes expose a massive right-side tail populated by heavy outlier data points. This highlights that Waze relies on a distinct segment of **Power Users** (likely professional drivers, daily cross-city commuters, or logistics drivers) whose extreme numbers shift raw arithmetic means upward.

### 💡 Insight D: Platform Uniformity in Churn Rates
Segmenting churn habits across mobile device architectures proves that churn behavior is platform-neutral:
* **iPhone Churn Share:** **17.83%** ($1,645$ churned / $9,225$ total)
* **Android Churn Share:** **17.56%** ($891$ churned / $5,074$ total)

Because the attrition percentages are nearly identical across both operating systems, user loss is driven by general behavioral retention factors rather than phone-specific technical bugs or platform optimization errors.

---

## 3️⃣ Next Steps for Modeling

1. **Feature Engineering:** Create custom ratio features to highlight user consistency, such as computing a *sessions-per-active-day* ratio (`sessions / activity_days`).
2. **Handling Class Imbalance:** Address the **82% vs. 18%** class skew by applying stratified data splits or leveraging oversampling/undersampling techniques.
3. **Algorithm Deployment:** Utilize robust tree-based machine learning ensembles (e.g., Random Forests, Gradient Boosting, or XGBoost) that can inherently prioritize the critical cutoff threshold of `activity_days` to capture high-risk drivers before they leave the platform.
